In [2]:

import numpy as np
import pandas as pd

np.random.seed(42)

# Generate 365 days of sample demand
days = np.arange(365)

trend = 20 + 0.02 * days
weekly_pattern = 3 * np.sin(2 * np.pi * days / 7)
seasonality = 4 * np.sin(2 * np.pi * days / 30)
noise = np.random.normal(0, 2, 365)

demand = np.maximum(
    1, np.round(trend + weekly_pattern + seasonality + noise)
).astype(int)

df = pd.DataFrame({
    "day": days + 1,
    "demand": demand
})

# Create features using past observations only
df["previous_day_demand"] = df["demand"].shift(1)
df["previous_week_demand"] = df["demand"].shift(7)
df["rolling_7_day_avg"] = (
    df["demand"].shift(1).rolling(7).mean()
)

df = df.dropna().reset_index(drop=True)

df.head(10)


,day,demand,previous_day_demand,previous_week_demand,rolling_7_day_avg
0,8,26,25.0,21.0,23.285714
1,9,26,26.0,23.0,24.000000
2,10,28,26.0,26.0,24.428571
3,11,24,28.0,27.0,24.714286
4,12,21,24.0,21.0,24.285714
5,13,20,21.0,20.0,24.285714
6,14,16,20.0,25.0,24.285714
7,15,18,16.0,26.0,23.000000
8,16,22,18.0,26.0,21.857143
9,17,20,22.0,28.0,21.285714


In [3]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import joblib

features = [
    "previous_day_demand",
    "previous_week_demand",
    "rolling_7_day_avg"
]

X = df[features]
y = df["demand"]

# Chronological split: earlier days for training,
# later days for testing
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

print(f"Test MAE: {mae:.2f} units per day")

# Save model for the Streamlit application
joblib.dump(model, "demand_model.joblib")

print("Model saved successfully!")


Test MAE: 2.22 units per day
Model saved successfully!


In [4]:

latest = df.iloc[-1]

next_day_input = pd.DataFrame([{
    "previous_day_demand": latest["demand"],
    "previous_week_demand": df.iloc[-7]["demand"],
    "rolling_7_day_avg": df["demand"].tail(7).mean()
}])

predicted_demand = model.predict(next_day_input)[0]

print(f"Predicted next-day demand: {max(0, predicted_demand):.1f} units")


Predicted next-day demand: 29.6 units


In [5]:

from google.colab import files

files.download("demand_model.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:

from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np

# Predictions from our trained ML model
ml_predictions = model.predict(X_test)

# Baseline: predict demand using previous day's demand
baseline_predictions = X_test["previous_day_demand"].to_numpy()

ml_mae = mean_absolute_error(y_test, ml_predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Linear Regression MAE: {ml_mae:.2f}")
print(f"Previous-day baseline MAE: {baseline_mae:.2f}")

improvement = (
    (baseline_mae - ml_mae) / baseline_mae * 100
    if baseline_mae != 0 else 0
)

print(f"MAE improvement over baseline: {improvement:.2f}%")

comparison = pd.DataFrame({
    "Actual demand": y_test.to_numpy(),
    "ML prediction": np.round(ml_predictions, 1),
    "Baseline prediction": baseline_predictions
})

comparison.head(10)


Linear Regression MAE: 2.22
Previous-day baseline MAE: 2.31
MAE improvement over baseline: 3.55%


,Actual demand,ML prediction,Baseline prediction
0,21,20.0,18.0
1,23,22.0,21.0
2,23,23.2,23.0
3,28,22.9,23.0
4,26,25.7,28.0
5,25,24.3,26.0
6,23,23.6,25.0
7,22,22.9,23.0
8,26,22.6,22.0
9,32,24.9,26.0


In [7]:

print(f"Linear Regression MAE: {ml_mae:.2f}")
print(f"Previous-day baseline MAE: {baseline_mae:.2f}")
print(f"MAE improvement over baseline: {improvement:.2f}%")


Linear Regression MAE: 2.22
Previous-day baseline MAE: 2.31
MAE improvement over baseline: 3.55%


In [8]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np
import joblib

# Add day-of-week as a feature
df["day_of_week"] = (df["day"] - 1) % 7

features_v2 = [
    "previous_day_demand",
    "previous_week_demand",
    "rolling_7_day_avg",
    "day_of_week"
]

X_v2 = df[features_v2]
y_v2 = df["demand"]

split_v2 = int(len(df) * 0.8)

X_train_v2 = X_v2.iloc[:split_v2]
X_test_v2 = X_v2.iloc[split_v2:]
y_train_v2 = y_v2.iloc[:split_v2]
y_test_v2 = y_v2.iloc[split_v2:]

model_v2 = LinearRegression()
model_v2.fit(X_train_v2, y_train_v2)

pred_v2 = model_v2.predict(X_test_v2)

mae_v2 = mean_absolute_error(y_test_v2, pred_v2)
baseline_mae_v2 = mean_absolute_error(
    y_test_v2,
    X_test_v2["previous_day_demand"]
)

print(f"New model MAE: {mae_v2:.2f}")
print(f"Baseline MAE: {baseline_mae_v2:.2f}")

joblib.dump(model_v2, "demand_model_v2.joblib")
print("New model saved!")


New model MAE: 2.09
Baseline MAE: 2.31
New model saved!


In [10]:

from google.colab import files

files.download("demand_model_v2.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>